# Working with the INbreast Dataset

This notebook demonstrates how to load and use the INbreast mammography dataset with the `medical_image` framework.

The `INbreastDataset` class automatically detects the annotation format:
- **COCO format** (`annotations.json` + `images/` folder)
- **XML format** (`AllDICOMs/` + `AllXML/` legacy layout)

In [ ]:
!pip install medical-image-std scikit-image

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from medical_image.datasets.inbreast import INbreastDataset

## 1. Dataset Structure

Our INbreast dataset uses a COCO-like layout:

```
data/Inbreast/
├── annotations.json     # COCO format (images, annotations, categories)
└── images/
    ├── 20586934.dcm
    ├── 53582683.dcm
    └── ...
```

The annotations contain polygon segmentations for **microcalcifications**.

## 2. Load the Dataset

In [ ]:
# Just point to the root directory — the dataset auto-detects COCO annotations
dataset = INbreastDataset(
    root_dir="data/Inbreast",
    target_size=(512, 512),
)

print(f"Dataset size: {len(dataset)} samples")
print(f"Mode: {dataset._mode}")

In [ ]:
# Access a single sample
sample = dataset[0]

image = sample["image"]
mask = sample["mask"]
meta = sample["metadata"]

print(f"Image shape: {image.shape}")
print(f"Mask shape:  {mask.shape}")
print(f"Image range: [{image.min():.1f}, {image.max():.1f}]")
print(f"Metadata:    {meta}")
print(f"Mask has lesions: {mask.sum().item() > 0}")

In [ ]:
# Visualize image and mask
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image.squeeze(0).numpy(), cmap="gray")
axes[0].set_title(f"Mammogram ({meta['case_id']})")
axes[1].imshow(mask.squeeze(0).numpy(), cmap="hot")
axes[1].set_title(f"MC Mask ({meta['num_annotations']} annotations)")
# Overlay
axes[2].imshow(image.squeeze(0).numpy(), cmap="gray")
axes[2].imshow(mask.squeeze(0).numpy(), cmap="Reds", alpha=0.4)
axes[2].set_title("Overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Using a DataLoader for Training

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0)

batch = next(iter(loader))
print(f"Batch image shape: {batch['image'].shape}")  # (4, 1, 512, 512)
print(f"Batch mask shape:  {batch['mask'].shape}")   # (4, 1, 512, 512)

In [ ]:
# Visualize a batch
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(4, batch["image"].shape[0])):
    axes[0, i].imshow(batch["image"][i, 0].numpy(), cmap="gray")
    axes[0, i].set_title(f"Image {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(batch["mask"][i, 0].numpy(), cmap="hot")
    axes[1, i].set_title(f"Mask {i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 4. Dataset Statistics

In [ ]:
# Annotation counts per image
ann_counts = [
    len(dataset._img_to_anns.get(img["id"], []))
    for img in dataset._samples
]

print(f"Total annotations: {sum(ann_counts)}")
print(f"Annotations per image: min={min(ann_counts)}, max={max(ann_counts)}, "
      f"mean={np.mean(ann_counts):.1f}")

plt.figure(figsize=(8, 3))
plt.hist(ann_counts, bins=20, edgecolor="black")
plt.xlabel("Annotations per image")
plt.ylabel("Count")
plt.title("Distribution of microcalcification annotations")
plt.tight_layout()
plt.show()

## 5. Applying Transforms

In [ ]:
import torchvision.transforms as T

# Normalize images to [-1, 1]
image_transform = T.Normalize(mean=[0.5], std=[0.5])

dataset_norm = INbreastDataset(
    root_dir="data/Inbreast",
    target_size=(256, 256),
    transform=image_transform,
)

sample = dataset_norm[0]
print(f"Normalized range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")

## 6. COCO JSON Utilities

In [ ]:
from medical_image.datasets.base_dataset import BaseDataset

# Parse COCO JSON with the framework utility
coco_data = BaseDataset.from_coco_json("data/Inbreast/annotations.json")

print(f"Images: {len(coco_data['images'])}")
print(f"Categories: {coco_data['categories']}")
print(f"Images with annotations: {len(coco_data['annotations'])}")